# ML-1M temporal skips — live in-kernel runner

No subprocesses and no hidden baseline evaluation. This prints stage timings and progress every 5 training batches.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, sys, shutil, subprocess, time, json, math, random
from pathlib import Path
REPO='/content/Sparsewalker'; BRANCH='agent/walker-temporal-skips'
if os.path.exists(REPO): shutil.rmtree(REPO)
subprocess.run(['git','clone','-q','-b',BRANCH,'https://github.com/hanialshater/Sparsewalker-.git',REPO],check=True)
SRC=f'{REPO}/src'; sys.path.insert(0,SRC)
for name in list(sys.modules):
    if name=='sparsewalker' or name.startswith('sparsewalker.'): del sys.modules[name]

import numpy as np, pandas as pd, torch
from torch.utils.data import DataLoader
from sparsewalker.data import load_dataset, split_data, WindowDataset, collate_windows
from sparsewalker.models import SparseWalkerTemporalMemory
from sparsewalker.models.core import ar_training_loss
from sparsewalker.training.trainer import _length_bucket_batches
from sparsewalker.evaluation import evaluate_full

assert torch.cuda.is_available(); device=torch.device('cuda')
torch.manual_seed(42); np.random.seed(42); random.seed(42)
print('GPU',torch.cuda.get_device_name(0),'bf16',torch.cuda.is_bf16_supported(),flush=True)

t=time.perf_counter(); print('LOAD DATA start',flush=True)
data=load_dataset('ml1m','/content/sparsewalker_data'); split=split_data(data['sequences'])
print('LOAD DATA done',round(time.perf_counter()-t,2),'s','users',len(data['sequences']),'items',data['n_items'],flush=True)

base_path=Path('/content/drive/MyDrive/sparsewalker_canonical_pair/ml1m/seed42/SparseWalker_FullCE/best.pt')
t=time.perf_counter(); print('LOAD CKPT start',base_path,flush=True)
base_ckpt=torch.load(base_path,map_location='cpu')
base_val=float(base_ckpt.get('val',{}).get('NDCG@10',float('nan')))
print('LOAD CKPT done',round(time.perf_counter()-t,2),'s','base_epoch',base_ckpt.get('epoch'),'base_val',base_val,flush=True)

model=SparseWalkerTemporalMemory(data['n_items'],200,d=64,layers=2,side=256,h=16,active=8,top_side=2,degree=4,fresh_weight=.25,memory_periods=(16,64,256),memory_offsets=(0,16,64),initial_memory_share=.25).to(device)
missing,unexpected=model.load_state_dict(base_ckpt['model'],strict=False)
print('WARMSTART missing',missing,'unexpected',unexpected,flush=True)
opt=torch.optim.AdamW(model.parameters(),lr=5e-4,weight_decay=1e-4)
ds=WindowDataset(split['train'],200,42)
OUT=Path('/content/drive/MyDrive/sparsewalker_temporal_skips_live/ml1m/seed42'); OUT.mkdir(parents=True,exist_ok=True)
MAX_EPOCHS=12; EVAL_EVERY=2; history=[]; best=-1.; best_state=None; best_epoch=0
print('READY: starting live training',flush=True)

In [ ]:
def set_lr(epoch):
    peak=5e-4; min_lr=1e-4; warmup=2
    if epoch<=warmup: lr=peak*epoch/warmup
    else:
        p=(epoch-warmup)/max(1,MAX_EPOCHS-warmup)
        lr=min_lr+.5*(peak-min_lr)*(1+math.cos(math.pi*p))
    for g in opt.param_groups: g['lr']=lr
    return lr

for epoch in range(1,MAX_EPOCHS+1):
    ds.set_epoch(epoch); g=torch.Generator().manual_seed(ds.seed+epoch)
    batches=_length_bucket_batches(ds,128,g)
    loader=DataLoader(ds,batch_sampler=batches,collate_fn=collate_windows,pin_memory=True)
    lr=set_lr(epoch); model.train(); total=0.; n=0; positions=0
    torch.cuda.synchronize(); e0=time.perf_counter(); last=time.perf_counter()
    print(f'\nEPOCH {epoch} START batches={len(batches)} lr={lr:.6g}',flush=True)
    for bi,(tokens,lengths) in enumerate(loader,1):
        positions += int((lengths-1).clamp_min(0).sum())
        tokens=tokens.to(device,non_blocking=True); lengths=lengths.to(device,non_blocking=True)
        opt.zero_grad(set_to_none=True)
        with torch.autocast('cuda',dtype=torch.bfloat16):
            loss=ar_training_loss(model,tokens,lengths,loss_mode='full')
        loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),5.0); opt.step()
        total += float(loss.detach()); n += 1
        if bi==1 or bi%5==0 or bi==len(batches):
            torch.cuda.synchronize(); now=time.perf_counter()
            print({'epoch':epoch,'batch':bi,'of':len(batches),'batch_window_s':round(now-last,2),'elapsed_s':round(now-e0,2),'avg_loss':round(total/n,4),'batch_L':int(tokens.size(1))},flush=True)
            last=now
    torch.cuda.synchronize(); secs=time.perf_counter()-e0
    train_loss=total/max(1,n)
    print('EPOCH TRAIN DONE',{'epoch':epoch,'seconds':round(secs,2),'loss':train_loss,'positions_per_s':round(positions/secs,1),'memory_share':float(torch.sigmoid(model.memory_share_logit).detach())},flush=True)

    if epoch==1 or epoch%EVAL_EVERY==0:
        print('EVAL START',epoch,flush=True); torch.cuda.synchronize(); v0=time.perf_counter()
        val=evaluate_full(model,split['val_prefix'],split['val_target'],data['n_items'],200,device,topks=(10,),batch_size=1024)
        torch.cuda.synchronize(); vsec=time.perf_counter()-v0
        ndcg=float(val['NDCG@10']); row={'epoch':epoch,'loss':train_loss,'seconds':secs,'eval_seconds':vsec,**val}
        history.append(row); pd.DataFrame(history).to_csv(OUT/'history.csv',index=False)
        print('TEMPORAL EVAL',{'epoch':epoch,'NDCG@10':ndcg,'HR@10':float(val['HR@10']),'MRR@10':float(val['MRR@10']),'eval_seconds':round(vsec,2),'gain_vs_base_val_pct':100*(ndcg/base_val-1) if base_val==base_val else None},flush=True)
        if ndcg>best:
            best=ndcg; best_epoch=epoch; best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
            torch.save({'model':best_state,'epoch':epoch,'val':val,'protocol':base_ckpt.get('protocol',{})},OUT/'best.pt')
        torch.save({'model':{k:v.detach().cpu().clone() for k,v in model.state_dict().items()},'optimizer':opt.state_dict(),'epoch':epoch,'best_ndcg':best,'best_epoch':best_epoch,'history':history,'protocol':base_ckpt.get('protocol',{})},OUT/'last.pt')

print('TRAINING COMPLETE',{'best_epoch':best_epoch,'best_val_NDCG@10':best,'base_val_NDCG@10':base_val},flush=True)